# Mutual Fund Performance Analytics

Comprehensive performance analysis of 40 mutual fund schemes:
- Daily returns & distribution validation
- CAGR (1yr, 3yr, 5yr) comparison table
- Sharpe Ratio ranking
- Sortino Ratio
- Alpha & Beta (OLS regression vs Nifty 100)
- Maximum Drawdown
- Fund Scorecard (0-100 composite)
- Benchmark comparison chart (top 5 vs Nifty 50 & Nifty 100)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
from datetime import datetime, timedelta
import warnings
import os
warnings.filterwarnings('ignore')

# Ensure we always run from the workspace root (where this notebook's parent is)
_nb_dir = os.getcwd()
_workspace = '/Users/maneeshreddy/Desktop/mutual_fund_analysis'
if os.path.isdir(_workspace):
    os.chdir(_workspace)
print(f'Working directory: {os.getcwd()}')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:.4f}'.format)

DATA_DIR = 'data_1/Bluestock_MF_Datasets'
REPORTS_DIR = 'reports/figures'
RF = 0.065  # Risk-free rate (RBI repo rate proxy)
TRADING_DAYS = 252

print('Libraries loaded successfully.')

## 1. Load and Prepare Data

In [ ]:
# Load NAV history
nav_df = pd.read_csv(f'{DATA_DIR}/02_nav_history.csv', parse_dates=['date'])
print(f'NAV data shape: {nav_df.shape}')
print(f'Date range: {nav_df["date"].min()} to {nav_df["date"].max()}')
print(f'Unique schemes: {nav_df["amfi_code"].nunique()}')
nav_df.head()

In [ ]:
# Load fund master for metadata
fund_master = pd.read_csv(f'{DATA_DIR}/01_fund_master.csv')
print(f'Fund master shape: {fund_master.shape}')
fund_master.head()

In [ ]:
# Load scheme performance (for expense ratio etc.)
scheme_perf = pd.read_csv(f'{DATA_DIR}/07_scheme_performance.csv')
print(f'Scheme performance shape: {scheme_perf.shape}')
scheme_perf.head()

In [ ]:
# Load benchmark indices
benchmark_df = pd.read_csv(f'{DATA_DIR}/10_benchmark_indices.csv', parse_dates=['date'])
print(f'Benchmark data shape: {benchmark_df.shape}')
print(f'Available indices: {benchmark_df["index_name"].unique()}')
benchmark_df.head()

In [ ]:
# Pivot NAV data: dates as index, schemes as columns
nav_pivot = nav_df.pivot(index='date', columns='amfi_code', values='nav')
nav_pivot = nav_pivot.sort_index()
print(f'NAV pivot shape: {nav_pivot.shape}')
print(f'Columns (first 5): {list(nav_pivot.columns[:5])}')
nav_pivot.head()

In [ ]:
# Pivot benchmark data
benchmark_pivot = benchmark_df.pivot(index='date', columns='index_name', values='close_value')
benchmark_pivot = benchmark_pivot.sort_index()
print(f'Benchmark pivot shape: {benchmark_pivot.shape}')
print(f'Columns: {list(benchmark_pivot.columns)}')
benchmark_pivot.head()

In [ ]:
# Create a mapping from amfi_code to scheme name
code_to_name = dict(zip(fund_master['amfi_code'], fund_master['scheme_name']))
code_to_name_int = {int(k): v for k, v in code_to_name.items()}

# Map column names
nav_pivot.columns = [f'{code_to_name_int.get(c, c)} ({c})' for c in nav_pivot.columns]
print('Scheme names mapped.')
print(list(nav_pivot.columns[:5]))

## 2. Compute Daily Returns

Formula: `daily_return = nav_t / nav_{t-1} - 1`

In [ ]:
# Compute daily returns
daily_returns = nav_pivot.pct_change().dropna()
print(f'Daily returns shape: {daily_returns.shape}')
daily_returns.head(10)

In [ ]:
# Summary statistics of daily returns
daily_returns.describe().T.sort_values('mean', ascending=False).head(10)

In [ ]:
# Validate distribution: plot histograms for all funds
n_cols = 5
n_rows = (len(daily_returns.columns) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(25, 5 * n_rows))
axes = axes.flatten()

for i, col in enumerate(daily_returns.columns):
    ax = axes[i]
    data = daily_returns[col].dropna()
    ax.hist(data, bins=80, alpha=0.7, color='steelblue', edgecolor='white', linewidth=0.3)
    ax.axvline(0, color='red', linewidth=0.8, linestyle='--')
    ax.set_title(col[:45] + '...' if len(col) > 45 else col, fontsize=7)
    ax.tick_params(labelsize=6)
    
    # Add normality info
    mu, sigma = data.mean(), data.std()
    ax.text(0.02, 0.95, f'μ={mu:.4f}\nσ={sigma:.4f}', transform=ax.transAxes,
            fontsize=6, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Daily Return Distributions - All 40 Schemes', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{REPORTS_DIR}/10_daily_return_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Distribution histograms saved.')

In [ ]:
# Validate distribution: check for normality (Shapiro-Wilk on sample) and basic stats
dist_validation = pd.DataFrame()
for col in daily_returns.columns:
    data = daily_returns[col].dropna()
    # Use a sample for Shapiro-Wilk (max 5000)
    sample = data.sample(min(len(data), 5000), random_state=42)
    shapiro_stat, shapiro_p = stats.shapiro(sample)
    
    dist_validation.loc[col, 'mean'] = data.mean()
    dist_validation.loc[col, 'std'] = data.std()
    dist_validation.loc[col, 'skewness'] = data.skew()
    dist_validation.loc[col, 'kurtosis'] = data.kurtosis()
    dist_validation.loc[col, 'shapiro_stat'] = shapiro_stat
    dist_validation.loc[col, 'shapiro_p'] = shapiro_p
    dist_validation.loc[col, 'min'] = data.min()
    dist_validation.loc[col, 'max'] = data.max()
    dist_validation.loc[col, 'n_obs'] = len(data)

dist_validation = dist_validation.sort_values('std', ascending=False)
print('=== Distribution Validation Summary ===')
print(f"Mean daily return range: {dist_validation['mean'].min():.6f} to {dist_validation['mean'].max():.6f}")
print(f"Std dev range: {dist_validation['std'].min():.6f} to {dist_validation['std'].max():.6f}")
print(f"Skewness range: {dist_validation['skewness'].min():.4f} to {dist_validation['skewness'].max():.4f}")
print(f"Kurtosis range: {dist_validation['kurtosis'].min():.4f} to {dist_validation['kurtosis'].max():.4f}")
print(f"Normal (p>0.05): {(dist_validation['shapiro_p'] > 0.05).sum()} / {len(dist_validation)}")
print()
dist_validation.head(10)

In [ ]:
# Box plot of daily returns for visual validation
fig, ax = plt.subplots(figsize=(20, 8))
daily_returns.boxplot(ax=ax, rot=90, fontsize=7)
ax.set_title('Daily Returns Box Plot - All 40 Schemes', fontsize=14, fontweight='bold')
ax.set_ylabel('Daily Return')
ax.axhline(0, color='red', linewidth=0.8, linestyle='--')
plt.tight_layout()
plt.savefig(f'{REPORTS_DIR}/10b_daily_return_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Compute CAGR for 1yr, 3yr, 5yr

Formula: `CAGR = (NAV_end / NAV_start) ^ (1/n) - 1`

Note: Our data spans ~4.4 years (Jan 2022 - May 2026), so 5yr CAGR will be computed from the earliest available data. For funds without full 5yr history, we compute from their start date.

In [ ]:
# Define end date as the latest date in our data
end_date = nav_pivot.index.max()
print(f'End date: {end_date}')

# Define start dates for 1yr, 3yr, 5yr
start_1yr = end_date - pd.DateOffset(years=1)
start_3yr = end_date - pd.DateOffset(years=3)
start_5yr = end_date - pd.DateOffset(years=5)

print(f'1yr lookback: {start_1yr}')
print(f'3yr lookback: {start_3yr}')
print(f'5yr lookback: {start_5yr}')
print(f'Data starts: {nav_pivot.index.min()}')

In [ ]:
def compute_cagr(nav_series, years):
    """Compute CAGR given a NAV series and number of years."""
    nav_clean = nav_series.dropna()
    if len(nav_clean) < 2:
        return np.nan
    
    nav_start = nav_clean.iloc[0]
    nav_end = nav_clean.iloc[-1]
    
    if nav_start <= 0:
        return np.nan
    
    # Actual years based on data span
    actual_years = (nav_clean.index[-1] - nav_clean.index[0]).days / 365.25
    if actual_years < 0.5:  # Need at least 6 months
        return np.nan
    
    return (nav_end / nav_start) ** (1 / actual_years) - 1

# Compute CAGR for each period
cagr_results = pd.DataFrame()

for col in nav_pivot.columns:
    nav_series = nav_pivot[col].dropna()
    
    # 1yr CAGR
    mask_1yr = nav_series.index >= start_1yr
    nav_1yr = nav_series[mask_1yr]
    cagr_1yr = compute_cagr(nav_1yr, 1)
    
    # 3yr CAGR
    mask_3yr = nav_series.index >= start_3yr
    nav_3yr = nav_series[mask_3yr]
    cagr_3yr = compute_cagr(nav_3yr, 3)
    
    # 5yr CAGR (use all available data since we only have ~4.4 years)
    cagr_5yr = compute_cagr(nav_series, 5)
    
    cagr_results.loc[col, 'CAGR_1yr'] = cagr_1yr
    cagr_results.loc[col, 'CAGR_3yr'] = cagr_3yr
    cagr_results.loc[col, 'CAGR_5yr'] = cagr_5yr
    cagr_results.loc[col, 'data_start'] = nav_series.index.min()
    cagr_results.loc[col, 'data_end'] = nav_series.index.max()
    cagr_results.loc[col, 'n_days'] = len(nav_series)

cagr_results = cagr_results.sort_values('CAGR_3yr', ascending=False)
print('=== CAGR Comparison Table ===')
cagr_results[['CAGR_1yr', 'CAGR_3yr', 'CAGR_5yr']].head(40)

In [ ]:
# Visualize CAGR comparison
fig, axes = plt.subplots(1, 3, figsize=(24, 10))

for ax, col_name, title in zip(axes, 
                                ['CAGR_1yr', 'CAGR_3yr', 'CAGR_5yr'],
                                ['1-Year CAGR', '3-Year CAGR', '5-Year CAGR (Available Data)']):
    sorted_data = cagr_results[col_name].sort_values(ascending=True)
    colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in sorted_data.values]
    sorted_data.plot(kind='barh', ax=ax, color=colors, edgecolor='white', linewidth=0.5)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('CAGR (%)')
    ax.axvline(0, color='black', linewidth=0.5)
    ax.tick_params(labelsize=7)
    # Format x-axis as percentage
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x*100:.0f}%'))

plt.suptitle('CAGR Comparison Across All Funds', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{REPORTS_DIR}/11_cagr_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Sharpe Ratio

Formula: `Sharpe = (Rp - Rf) / Std(Rp) × √252`

Where Rf = 6.5% (RBI repo rate proxy)

In [ ]:
# Compute Sharpe Ratio for each fund
sharpe_results = pd.DataFrame()

for col in daily_returns.columns:
    returns = daily_returns[col].dropna()
    
    if len(returns) < 60:  # Need minimum data
        continue
    
    excess_return = returns.mean() - RF / TRADING_DAYS
    annualized_sharpe = (excess_return / returns.std()) * np.sqrt(TRADING_DAYS)
    
    sharpe_results.loc[col, 'mean_daily_return'] = returns.mean()
    sharpe_results.loc[col, 'std_daily_return'] = returns.std()
    sharpe_results.loc[col, 'annualized_return'] = returns.mean() * TRADING_DAYS
    sharpe_results.loc[col, 'annualized_std'] = returns.std() * np.sqrt(TRADING_DAYS)
    sharpe_results.loc[col, 'sharpe_ratio'] = annualized_sharpe

sharpe_results = sharpe_results.sort_values('sharpe_ratio', ascending=False)
sharpe_results['sharpe_rank'] = range(1, len(sharpe_results) + 1)

print('=== Sharpe Ratio Ranking ===')
print(f'Risk-free rate: {RF*100}%')
print()
sharpe_results[['annualized_return', 'annualized_std', 'sharpe_ratio', 'sharpe_rank']]

In [ ]:
# Visualize Sharpe Ratio
fig, ax = plt.subplots(figsize=(16, 12))
sorted_sharpe = sharpe_results['sharpe_ratio'].sort_values(ascending=True)
colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in sorted_sharpe.values]
sorted_sharpe.plot(kind='barh', ax=ax, color=colors, edgecolor='white', linewidth=0.5)
ax.set_title('Sharpe Ratio Ranking - All 40 Funds\n(Rf = 6.5%, Annualized)', fontsize=14, fontweight='bold')
ax.set_xlabel('Sharpe Ratio')
ax.axvline(0, color='black', linewidth=0.8)
ax.axvline(1, color='orange', linewidth=0.8, linestyle='--', label='Sharpe = 1.0')
ax.legend(fontsize=11)
ax.tick_params(labelsize=8)
plt.tight_layout()
plt.savefig(f'{REPORTS_DIR}/12_sharpe_ratio_ranking.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Sortino Ratio

Formula: `Sortino = (Rp - Rf) / Std(downside returns) × √252`

Denominator uses only negative return days (downside deviation).

In [ ]:
# Compute Sortino Ratio for each fund
sortino_results = pd.DataFrame()

for col in daily_returns.columns:
    returns = daily_returns[col].dropna()
    
    if len(returns) < 60:
        continue
    
    excess_return = returns.mean() - RF / TRADING_DAYS
    
    # Downside deviation: std of negative returns only
    downside_returns = returns[returns < 0]
    
    if len(downside_returns) == 0:
        sortino = np.inf
    else:
        downside_std = downside_returns.std()
        sortino = (excess_return / downside_std) * np.sqrt(TRADING_DAYS)
    
    sortino_results.loc[col, 'mean_daily_return'] = returns.mean()
    sortino_results.loc[col, 'downside_std'] = downside_returns.std() if len(downside_returns) > 0 else np.nan
    sortino_results.loc[col, 'pct_negative_days'] = len(downside_returns) / len(returns) * 100
    sortino_results.loc[col, 'sortino_ratio'] = sortino

sortino_results = sortino_results.sort_values('sortino_ratio', ascending=False)
sortino_results['sortino_rank'] = range(1, len(sortino_results) + 1)

print('=== Sortino Ratio Ranking ===')
sortino_results[['mean_daily_return', 'downside_std', 'pct_negative_days', 'sortino_ratio', 'sortino_rank']]

In [ ]:
# Compare Sharpe vs Sortino
fig, ax = plt.subplots(figsize=(14, 10))
comparison = pd.DataFrame({
    'Sharpe': sharpe_results['sharpe_ratio'],
    'Sortino': sortino_results['sortino_ratio']
}).dropna()

x = np.arange(len(comparison))
width = 0.35
bars1 = ax.barh(x - width/2, comparison['Sharpe'], width, label='Sharpe', color='#3498db', alpha=0.8)
bars2 = ax.barh(x + width/2, comparison['Sortino'], width, label='Sortino', color='#e67e22', alpha=0.8)
ax.set_yticks(x)
ax.set_yticklabels([c[:50] for c in comparison.index], fontsize=7)
ax.set_xlabel('Ratio Value')
ax.set_title('Sharpe vs Sortino Ratio Comparison', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.axvline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.savefig(f'{REPORTS_DIR}/13_sharpe_vs_sortino.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Alpha and Beta (OLS Regression)

OLS regression of fund returns on Nifty 100 returns using `scipy.stats.linregress`.
- Alpha = intercept × 252
- Beta = slope

In [ ]:
# Compute benchmark (Nifty 100) daily returns
nifty100_returns = benchmark_pivot['NIFTY100'].pct_change().dropna()
nifty50_returns = benchmark_pivot['NIFTY50'].pct_change().dropna()

print(f'Nifty 100 returns: {len(nifty100_returns)} observations')
print(f'Nifty 50 returns: {len(nifty50_returns)} observations')
print(f'Fund returns: {len(daily_returns)} observations')

In [ ]:
# Run OLS regression for each fund against Nifty 100
alpha_beta_results = pd.DataFrame()

for col in daily_returns.columns:
    fund_returns = daily_returns[col].dropna()
    
    # Align dates with Nifty 100
    common_dates = fund_returns.index.intersection(nifty100_returns.index)
    
    if len(common_dates) < 60:
        continue
    
    y = fund_returns.loc[common_dates].values
    x = nifty100_returns.loc[common_dates].values
    
    # OLS regression using scipy.stats.linregress
    slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
    
    alpha_beta_results.loc[col, 'alpha_annualized'] = intercept * TRADING_DAYS
    alpha_beta_results.loc[col, 'beta'] = slope
    alpha_beta_results.loc[col, 'r_squared'] = r_value ** 2
    alpha_beta_results.loc[col, 'p_value'] = p_value
    alpha_beta_results.loc[col, 'std_err'] = std_err
    alpha_beta_results.loc[col, 'n_obs'] = len(common_dates)

alpha_beta_results = alpha_beta_results.sort_values('alpha_annualized', ascending=False)
alpha_beta_results['alpha_rank'] = range(1, len(alpha_beta_results) + 1)

print('=== Alpha & Beta Results (vs Nifty 100) ===')
alpha_beta_results[['alpha_annualized', 'beta', 'r_squared', 'p_value', 'alpha_rank']]

In [ ]:
# Save alpha_beta.csv
alpha_beta_results.to_csv('alpha_beta.csv')
print('alpha_beta.csv saved.')

In [ ]:
# Visualize Alpha vs Beta
fig, ax = plt.subplots(figsize=(14, 10))
scatter = ax.scatter(
    alpha_beta_results['beta'],
    alpha_beta_results['alpha_annualized'] * 100,
    c=alpha_beta_results['r_squared'],
    cmap='RdYlGn',
    s=100,
    edgecolors='black',
    linewidth=0.5
)
plt.colorbar(scatter, label='R²')

# Label each point
for idx, row in alpha_beta_results.iterrows():
    short_name = idx.split('(')[0].strip()[:25]
    ax.annotate(short_name, (row['beta'], row['alpha_annualized'] * 100),
                fontsize=6, alpha=0.8, xytext=(3, 3), textcoords='offset points')

ax.axhline(0, color='red', linewidth=0.8, linestyle='--', alpha=0.5)
ax.axvline(1, color='blue', linewidth=0.8, linestyle='--', alpha=0.5, label='Market Beta = 1')
ax.set_xlabel('Beta', fontsize=12)
ax.set_ylabel('Alpha (Annualized %)', fontsize=12)
ax.set_title('Alpha vs Beta (vs Nifty 100)\nColor = R²', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig(f'{REPORTS_DIR}/14_alpha_beta_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Maximum Drawdown

Formula: `Max DD = min(NAV / running_max - 1)`

Also find the worst drawdown date range for each fund.

In [ ]:
# Compute Maximum Drawdown for each fund
drawdown_results = pd.DataFrame()

for col in nav_pivot.columns:
    nav_series = nav_pivot[col].dropna()
    
    if len(nav_series) < 2:
        continue
    
    # Running maximum
    running_max = nav_series.cummax()
    
    # Drawdown series
    drawdown = (nav_series / running_max - 1)
    
    # Maximum drawdown
    max_dd = drawdown.min()
    max_dd_date = drawdown.idxmin()
    
    # Find the peak before the max drawdown
    peak_before_dd = nav_series.loc[:max_dd_date].idxmax()
    
    # Find recovery date (if any)
    nav_at_peak = nav_series.loc[peak_before_dd]
    post_dd = nav_series.loc[max_dd_date:]
    recovery_dates = post_dd[post_dd >= nav_at_peak]
    recovery_date = recovery_dates.index[0] if len(recovery_dates) > 0 else None
    
    drawdown_results.loc[col, 'max_drawdown'] = max_dd
    drawdown_results.loc[col, 'max_dd_date'] = max_dd_date
    drawdown_results.loc[col, 'peak_date'] = peak_before_dd
    drawdown_results.loc[col, 'recovery_date'] = recovery_date
    drawdown_results.loc[col, 'drawdown_duration_days'] = (max_dd_date - peak_before_dd).days
    
    if recovery_date:
        drawdown_results.loc[col, 'total_duration_days'] = (recovery_date - peak_before_dd).days
    else:
        drawdown_results.loc[col, 'total_duration_days'] = (nav_series.index[-1] - peak_before_dd).days

drawdown_results = drawdown_results.sort_values('max_drawdown', ascending=True)
drawdown_results['dd_rank'] = range(1, len(drawdown_results) + 1)

print('=== Maximum Drawdown Results ===')
drawdown_results[['max_drawdown', 'peak_date', 'max_dd_date', 'recovery_date', 'drawdown_duration_days', 'dd_rank']]

In [ ]:
# Plot drawdown curves for all funds
fig, ax = plt.subplots(figsize=(18, 10))

for col in nav_pivot.columns:
    nav_series = nav_pivot[col].dropna()
    running_max = nav_series.cummax()
    drawdown = (nav_series / running_max - 1) * 100
    ax.plot(drawdown.index, drawdown.values, alpha=0.5, linewidth=0.7, label=col[:30])

ax.set_title('Drawdown Curves - All 40 Funds', fontsize=14, fontweight='bold')
ax.set_ylabel('Drawdown (%)')
ax.set_xlabel('Date')
ax.axhline(0, color='black', linewidth=0.8)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(f'{REPORTS_DIR}/15_drawdown_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Visualize max drawdown ranking
fig, ax = plt.subplots(figsize=(16, 12))
sorted_dd = drawdown_results['max_drawdown'].sort_values(ascending=False)
colors = ['#e74c3c' for _ in sorted_dd]
sorted_dd.plot(kind='barh', ax=ax, color=colors, edgecolor='white', linewidth=0.5)
ax.set_title('Maximum Drawdown - All 40 Funds', fontsize=14, fontweight='bold')
ax.set_xlabel('Max Drawdown (%)')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x*100:.0f}%'))
ax.tick_params(labelsize=8)
plt.tight_layout()
plt.savefig(f'{REPORTS_DIR}/15b_max_drawdown_ranking.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Fund Scorecard (0-100)

Composite score:
- 30% × 3yr CAGR rank
- 25% × Sharpe ratio rank
- 20% × Alpha rank
- 15% × Expense ratio rank (inverse - lower is better)
- 10% × Max drawdown rank (inverse - lower is better)

In [ ]:
# Get expense ratio from scheme_perf
# Map amfi_code to expense_ratio_pct
expense_map = dict(zip(scheme_perf['amfi_code'], scheme_perf['expense_ratio_pct']))

# Extract amfi_code from the column names (in parentheses)
def extract_amfi_code(col_name):
    import re
    match = re.search(r'\((\d+)\)', col_name)
    return int(match.group(1)) if match else None

expense_ratios = {}
for col in nav_pivot.columns:
    code = extract_amfi_code(col)
    if code and code in expense_map:
        expense_ratios[col] = expense_map[code]
    else:
        expense_ratios[col] = np.nan

print('Expense ratios mapped:')
for k, v in list(expense_ratios.items())[:5]:
    print(f'  {k[:50]}: {v}%')

In [ ]:
# Build the composite scorecard
scorecard = pd.DataFrame(index=nav_pivot.columns)

# 1. 3yr CAGR rank (higher is better)
scorecard['cagr_3yr'] = cagr_results['CAGR_3yr']
scorecard['cagr_3yr_rank'] = scorecard['cagr_3yr'].rank(ascending=False, method='min')

# 2. Sharpe ratio rank (higher is better)
scorecard['sharpe_ratio'] = sharpe_results['sharpe_ratio']
scorecard['sharpe_rank'] = scorecard['sharpe_ratio'].rank(ascending=False, method='min')

# 3. Alpha rank (higher is better)
scorecard['alpha_annualized'] = alpha_beta_results['alpha_annualized']
scorecard['alpha_rank'] = scorecard['alpha_annualized'].rank(ascending=False, method='min')

# 4. Expense ratio rank (lower is better, so inverse)
scorecard['expense_ratio'] = pd.Series(expense_ratios)
scorecard['expense_rank_inverse'] = scorecard['expense_ratio'].rank(ascending=True, method='min')

# 5. Max drawdown rank (lower is better, so inverse)
scorecard['max_drawdown'] = drawdown_results['max_drawdown']
scorecard['dd_rank_inverse'] = scorecard['max_drawdown'].rank(ascending=False, method='min')

# Normalize ranks to 0-100 scale
n_funds = len(scorecard)

def rank_to_score(rank_series, n):
    """Convert rank (1=best) to 0-100 score (100=best)."""
    return (n - rank_series + 1) / n * 100

scorecard['cagr_3yr_score'] = rank_to_score(scorecard['cagr_3yr_rank'], n_funds)
scorecard['sharpe_score'] = rank_to_score(scorecard['sharpe_rank'], n_funds)
scorecard['alpha_score'] = rank_to_score(scorecard['alpha_rank'], n_funds)
scorecard['expense_score'] = rank_to_score(scorecard['expense_rank_inverse'], n_funds)
scorecard['dd_score'] = rank_to_score(scorecard['dd_rank_inverse'], n_funds)

# Composite score
scorecard['composite_score'] = (
    0.30 * scorecard['cagr_3yr_score'] +
    0.25 * scorecard['sharpe_score'] +
    0.20 * scorecard['alpha_score'] +
    0.15 * scorecard['expense_score'] +
    0.10 * scorecard['dd_score']
)

scorecard = scorecard.sort_values('composite_score', ascending=False)
scorecard['overall_rank'] = range(1, len(scorecard) + 1)

print('=== Fund Scorecard (0-100) ===')
print(f'Weights: 30% 3yr CAGR + 25% Sharpe + 20% Alpha + 15% Expense (inv) + 10% Max DD (inv)')
print()
scorecard[['cagr_3yr', 'sharpe_ratio', 'alpha_annualized', 'expense_ratio', 'max_drawdown',
           'cagr_3yr_score', 'sharpe_score', 'alpha_score', 'expense_score', 'dd_score',
           'composite_score', 'overall_rank']]

In [ ]:
# Save fund_scorecard.csv
scorecard.to_csv('fund_scorecard.csv')
print('fund_scorecard.csv saved.')

In [ ]:
# Visualize scorecard
fig, ax = plt.subplots(figsize=(18, 12))
top_cols = ['cagr_3yr_score', 'sharpe_score', 'alpha_score', 'expense_score', 'dd_score']
scorecard_plot = scorecard[top_cols].copy()
scorecard_plot.columns = ['3yr CAGR\n(30%)', 'Sharpe\n(25%)', 'Alpha\n(20%)', 'Expense\n(15%)', 'Max DD\n(10%)']

scorecard_plot.plot(kind='barh', stacked=True, ax=ax, 
                    colormap='RdYlGn', edgecolor='white', linewidth=0.5)
ax.set_title('Fund Scorecard Breakdown (0-100 Composite)', fontsize=14, fontweight='bold')
ax.set_xlabel('Composite Score')
ax.tick_params(labelsize=7)
ax.legend(loc='lower right', fontsize=9)

# Add composite score labels
for i, (idx, row) in enumerate(scorecard.iterrows()):
    ax.text(row['composite_score'] + 0.5, i, f"{row['composite_score']:.1f}",
            va='center', fontsize=7, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{REPORTS_DIR}/16_fund_scorecard.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Top 10 funds summary
print('=== TOP 10 FUNDS BY COMPOSITE SCORE ===')
print()
top10 = scorecard.head(10)[['cagr_3yr', 'sharpe_ratio', 'alpha_annualized', 'expense_ratio', 'max_drawdown', 'composite_score']]
top10.columns = ['3yr CAGR', 'Sharpe', 'Alpha (ann.)', 'Expense %', 'Max DD', 'Score']
top10

## 9. Benchmark Comparison Chart

Plot top 5 funds vs Nifty 50 and Nifty 100 over 3 years.
Compute tracking error = std(fund_return - benchmark_return) × √252.

In [ ]:
# Get top 5 funds by composite score
top5_funds = scorecard.head(5).index.tolist()
print('Top 5 funds for benchmark comparison:')
for i, f in enumerate(top5_funds, 1):
    print(f'  {i}. {f[:60]} (Score: {scorecard.loc[f, "composite_score"]:.1f})')

In [ ]:
# Filter data to 3-year window
start_3yr_date = end_date - pd.DateOffset(years=3)
print(f'3-year window: {start_3yr_date} to {end_date}')

# Compute cumulative returns for top 5 funds
nav_3yr = nav_pivot.loc[start_3yr_date:]
cumulative_returns = pd.DataFrame()

for fund in top5_funds:
    if fund in nav_3yr.columns:
        normalized = nav_3yr[fund].dropna()
        cumulative_returns[fund] = (normalized / normalized.iloc[0] - 1) * 100

# Compute cumulative returns for benchmarks
for bench in ['NIFTY50', 'NIFTY100']:
    if bench in benchmark_pivot.columns:
        bench_data = benchmark_pivot.loc[start_3yr_date:, bench].dropna()
        cumulative_returns[bench] = (bench_data / bench_data.iloc[0] - 1) * 100

print(f'Cumulative returns shape: {cumulative_returns.shape}')
cumulative_returns.tail()

In [ ]:
# Plot benchmark comparison
fig, ax = plt.subplots(figsize=(18, 10))

colors = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6', '#e67e22', '#1abc9c', '#34495e']
linestyles = ['-', '-', '-', '-', '-', '--', '--']
linewidths = [2, 2, 2, 2, 2, 3, 3]

for i, col in enumerate(cumulative_returns.columns):
    short_name = col.split('(')[0].strip() if '(' in col else col
    is_bench = col in ['NIFTY50', 'NIFTY100']
    label = f'📊 {short_name}' if is_bench else short_name
    
    ax.plot(cumulative_returns.index, cumulative_returns[col],
            color=colors[i % len(colors)],
            linestyle='--' if is_bench else '-',
            linewidth=3 if is_bench else 2,
            alpha=0.9,
            label=label)

ax.set_title('Top 5 Funds vs Nifty 50 & Nifty 100 - 3 Year Cumulative Returns',
             fontsize=15, fontweight='bold')
ax.set_ylabel('Cumulative Return (%)', fontsize=12)
ax.set_xlabel('Date', fontsize=12)
ax.axhline(0, color='gray', linewidth=0.8, linestyle=':')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45)
ax.legend(loc='upper left', fontsize=9, framealpha=0.9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{REPORTS_DIR}/17_benchmark_comparison_3yr.png', dpi=200, bbox_inches='tight')
plt.savefig('benchmark_comparison_3yr.png', dpi=200, bbox_inches='tight')
plt.show()
print('Benchmark comparison chart saved.')

In [ ]:
# Compute Tracking Error for top 5 funds vs Nifty 100 and Nifty 50
tracking_error_results = pd.DataFrame()

for fund in top5_funds:
    if fund not in daily_returns.columns:
        continue
    fund_ret = daily_returns[fund].dropna()
    
    for bench_name, bench_ret in [('NIFTY100', nifty100_returns), ('NIFTY50', nifty50_returns)]:
        common = fund_ret.index.intersection(bench_ret.index)
        common = common[common >= start_3yr_date]
        
        if len(common) < 30:
            continue
        
        diff = fund_ret.loc[common] - bench_ret.loc[common]
        tracking_error = diff.std() * np.sqrt(TRADING_DAYS)
        
        tracking_error_results.loc[fund, f'TE_vs_{bench_name}'] = tracking_error
        tracking_error_results.loc[fund, f'corr_with_{bench_name}'] = fund_ret.loc[common].corr(bench_ret.loc[common])
        tracking_error_results.loc[fund, f'avg_excess_return_{bench_name}'] = diff.mean() * TRADING_DAYS

print('=== Tracking Error Analysis (Top 5 Funds, 3-Year) ===')
print(f'Tracking Error = std(fund_return - benchmark_return) × √252')
print()
tracking_error_results

In [ ]:
# Visualize tracking error
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, bench in zip(axes, ['NIFTY100', 'NIFTY50']):
    te_col = f'TE_vs_{bench}'
    if te_col in tracking_error_results.columns:
        te_data = tracking_error_results[te_col].sort_values(ascending=True)
        te_data.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
        ax.set_title(f'Tracking Error vs {bench}', fontsize=13, fontweight='bold')
        ax.set_xlabel('Tracking Error (Annualized)')
        ax.tick_params(labelsize=8)
        
        for i, v in enumerate(te_data.values):
            ax.text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=9)

plt.suptitle('Tracking Error - Top 5 Funds (3-Year)', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{REPORTS_DIR}/18_tracking_error.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Summary Dashboard

In [ ]:
# Create a comprehensive summary table
summary = pd.DataFrame()

# Add fund metadata
for col in scorecard.index:
    code = extract_amfi_code(col)
    
    # Get category and fund house from fund_master
    if code and code in fund_master['amfi_code'].values:
        fm_row = fund_master[fund_master['amfi_code'] == code].iloc[0]
        summary.loc[col, 'Fund House'] = fm_row['fund_house']
        summary.loc[col, 'Category'] = fm_row['category']
    
    summary.loc[col, '3yr CAGR'] = scorecard.loc[col, 'cagr_3yr']
    summary.loc[col, 'Sharpe'] = scorecard.loc[col, 'sharpe_ratio']
    summary.loc[col, 'Sortino'] = sortino_results.loc[col, 'sortino_ratio'] if col in sortino_results.index else np.nan
    summary.loc[col, 'Alpha (ann.)'] = scorecard.loc[col, 'alpha_annualized']
    summary.loc[col, 'Beta'] = alpha_beta_results.loc[col, 'beta'] if col in alpha_beta_results.index else np.nan
    summary.loc[col, 'Max DD'] = scorecard.loc[col, 'max_drawdown']
    summary.loc[col, 'Expense %'] = scorecard.loc[col, 'expense_ratio']
    summary.loc[col, 'Composite Score'] = scorecard.loc[col, 'composite_score']
    summary.loc[col, 'Rank'] = scorecard.loc[col, 'overall_rank']

summary = summary.sort_values('Composite Score', ascending=False)

print('=== COMPREHENSIVE PERFORMANCE SUMMARY ===')
print(f'Total funds analyzed: {len(summary)}')
print(f'Analysis period: {nav_pivot.index.min().date()} to {nav_pivot.index.max().date()}')
print(f'Risk-free rate: {RF*100}%')
print()
summary.head(40)

In [ ]:
# Final visualization: Risk-Return scatter
fig, ax = plt.subplots(figsize=(16, 12))

for idx, row in summary.iterrows():
    if pd.notna(row['Sharpe']) and pd.notna(row['3yr CAGR']):
        # Color by composite score
        score = row['Composite Score']
        color = plt.cm.RdYlGn(score / 100)
        
        ax.scatter(row['Sharpe'], row['3yr CAGR'] * 100,
                  c=[color], s=120, edgecolors='black', linewidth=0.5, zorder=5)
        
        short_name = idx.split('(')[0].strip()[:20]
        ax.annotate(short_name, (row['Sharpe'], row['3yr CAGR'] * 100),
                    fontsize=6, alpha=0.8, xytext=(3, 3), textcoords='offset points')

# Add benchmark points
for bench_name, bench_ret in [('NIFTY100', nifty100_returns), ('NIFTY50', nifty50_returns)]:
    bench_sharpe = ((bench_ret.mean() - RF/TRADING_DAYS) / bench_ret.std()) * np.sqrt(TRADING_DAYS)
    bench_cagr = bench_ret.mean() * TRADING_DAYS
    ax.scatter(bench_sharpe, bench_cagr * 100, c='red', s=200, marker='*',
              edgecolors='black', linewidth=1, zorder=10, label=bench_name)

ax.axhline(0, color='gray', linewidth=0.5, linestyle=':')
ax.axvline(0, color='gray', linewidth=0.5, linestyle=':')
ax.set_xlabel('Sharpe Ratio', fontsize=12)
ax.set_ylabel('Annualized Return (3yr CAGR %)', fontsize=12)
ax.set_title('Risk-Return Profile (Color = Composite Score)\nStars = Benchmarks',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig(f'{REPORTS_DIR}/19_risk_return_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print('=' * 80)
print('ANALYSIS COMPLETE')
print('=' * 80)
print()
print('Deliverables:')
print('  1. Performance_Analytics.ipynb - This notebook')
print('  2. fund_scorecard.csv - Composite scores for all 40 funds')
print('  3. alpha_beta.csv - Alpha and Beta from OLS regression vs Nifty 100')
print('  4. benchmark_comparison_3yr.png - Top 5 funds vs Nifty 50 & Nifty 100')
print()
print('Additional outputs in reports/figures/:')
print('  - 10_daily_return_distributions.png')
print('  - 10b_daily_return_boxplots.png')
print('  - 11_cagr_comparison.png')
print('  - 12_sharpe_ratio_ranking.png')
print('  - 13_sharpe_vs_sortino.png')
print('  - 14_alpha_beta_scatter.png')
print('  - 15_drawdown_curves.png')
print('  - 15b_max_drawdown_ranking.png')
print('  - 16_fund_scorecard.png')
print('  - 17_benchmark_comparison_3yr.png')
print('  - 18_tracking_error.png')
print('  - 19_risk_return_scatter.png')